In [1]:
from pathlib import Path
import re
import shutil
import subprocess
import sys

import ipywidgets as widgets
from Bio import SeqIO
from IPython.display import display, clear_output

def localizar_projeto():
    pasta = Path.cwd().resolve()
    for candidato in (pasta, *pasta.parents):
        if (candidato / "atividade").is_dir() and (candidato / "dados").is_dir():
            return candidato
    raise FileNotFoundError("Abra o JupyterLab na raiz do projeto ou em atividade/.")

PROJECT = localizar_projeto()
DATA_DIR = PROJECT / "dados"
RESULT_DIR = PROJECT / "resultados"
RESULT_DIR.mkdir(exist_ok=True)
sys.path.insert(0, str(PROJECT / "scripts"))
from alinhamentos import visualizador, arquivos_de_alinhamento

COMBINED = RESULT_DIR / "16S_com_isolado.fna"      # 229 sequências + as da turma
MAFFT_OUT = RESULT_DIR / "16S_com_isolado.aln"     # alinhamento feito aqui
print("Projeto:", PROJECT)


Projeto: /home/marialuiza/Luiza/Colabs/bacillus/migração_git/claude/Bacillotav4


# Explorando relações filogenéticas

Para fazer uma análise filogenética precisamos comparar características entre os organismos e o objetivo é traçar uma hipótese de como a diversificação de organismos se deu a partir de um ancestral em comum. Estamos tentando entender a **evolução** de um conjunto de organismos, no nosso caso, bactérias do filo Bacillota.

Portanto, precisamos escolher alguma característica para comparar os organismos, ver quais podem ter se diversificado a partir de um ancestral em comum mais recente do que os outros. Para começar a entender como escolher e o que escolher vamos começar com algo simples, vamos comparar um único gene.

Esse gene precisa:

- Estar presente em todos os organismos da minha amostra
- Ser um gene que permite ver diferenças na escala de tempo em que estamos trabalhando
- Precisamos que as sequências sejam **homólogas**
- Se quisermos reconstruir a história das _espécies_ precisamos que os genes escolhidos sejam **ortólogos** também

# 16S rRNA

O gene 16S rRNA é o primeiro marcador da atividade. As sequências dos **229 genomas de referência** já foram extraídas e estão em `dados/tree_16S.fna`. A elas vamos juntar as sequências do grupo: o 16S do isolado ambiental (o FASTA que vocês salvaram no Chromas) e as duas referências de *Bacillus*.

Clique abaixo para inspecionar as 229 sequências.


In [2]:
# As sequências já foram recuperadas e selecionadas na preparação da atividade.
# Isso evita depender da instalação do Bakta e dos arquivos .ffn no servidor DaVinci.
entrada_16S = DATA_DIR / 'tree_16S.fna'
btn_16s = widgets.Button(description='Carregar as sequências 16S', icon='folder-open', button_style='primary')
saida_16s = widgets.Output()
def carregar_16s(_):
    with saida_16s:
        clear_output(wait=True)
        try:
            seqs = list(SeqIO.parse(entrada_16S, 'fasta'))
            assert len(seqs) == 229, f'Esperávamos 229 sequências; encontradas {len(seqs)}.'
            tamanhos = [len(s.seq) for s in seqs]
            print(f'{len(seqs)} sequências de 16S recuperadas na curadoria prévia.')
            print(f'Tamanho mínimo: {min(tamanhos)} nt; máximo: {max(tamanhos)} nt.')
            print('Arquivo:', entrada_16S.relative_to(PROJECT))
            print('Exemplo de identificadores:', ', '.join(s.id for s in seqs[:3]))
        except Exception as e:
            print('Não foi possível ler o FASTA:', e)
btn_16s.on_click(carregar_16s)
display(widgets.VBox([btn_16s,saida_16s]))


## Juntando as sequências do grupo

Envie os FASTAs do grupo (o 16S do isolado e as duas referências). Use identificadores simples, só com letras, números, `_`, `.` e `-` (por exemplo `AMB_grupo01_16S`): o IQ-TREE troca outros caracteres, e o nome da ponta deixaria de bater com o seu.

O botão junta tudo com as 229 sequências em `resultados/16S_com_isolado.fna`.


In [3]:
# Carregando as sequências e unindo-as ao alinhamento anterior

upload = widgets.FileUpload(accept='.fasta,.fa,.fas,.fna,.txt', multiple=True,
                            description='FASTAs do grupo')
join_button = widgets.Button(description='Juntar com as 229', icon='plus', button_style='primary')
join_out = widgets.Output()
VALID_ID = re.compile(r'^[A-Za-z0-9_.-]+$')

def juntar(_):
    with join_out:
        clear_output(wait=True)
        if not upload.value:
            print('Envie pelo menos um FASTA.'); return
        base = list(SeqIO.parse(DATA_DIR / 'tree_16S.fna', 'fasta'))
        novas = []
        upload_dir = RESULT_DIR / 'fasta_grupo'
        upload_dir.mkdir(exist_ok=True)
        for f in upload.value:
            path = upload_dir / f['name']
            path.write_bytes(bytes(f['content']))
            recs = list(SeqIO.parse(path, 'fasta'))
            if not recs:
                print(f'{f["name"]}: nenhum registro FASTA encontrado.'); return
            novas += recs
        for rec in novas:
            rec.seq = rec.seq.replace('-', '').upper()
            rec.description = ''
        ids = [r.id for r in base + novas]
        dup = sorted({i for i in ids if ids.count(i) > 1})
        bad = [r.id for r in novas if not VALID_ID.match(r.id)]
        if dup:
            print('Identificadores repetidos:', ', '.join(dup)); return
        if bad:
            print('Troque estes identificadores (use só letras, números, _ . -):', ', '.join(bad)); return
        SeqIO.write(base + novas, COMBINED, 'fasta')
        for r in novas:
            print(f'+ {r.id}: {len(r.seq)} nt')
        print(f'\n{len(base) + len(novas)} sequências salvas em {COMBINED.relative_to(PROJECT)}')

join_button.on_click(juntar)
display(widgets.VBox([widgets.HBox([upload, join_button]), join_out]))


## Alinhamento com MAFFT

Para comparar regiões homólogas, as sequências precisam estar **alinhadas**. O botão abaixo roda o MAFFT no arquivo combinado e salva `resultados/16S_com_isolado.aln`. Também dá para usar o MAFFT online e salvar o resultado em `resultados/`.

Se algo der errado, há alinhamentos **pré-computados** em `dados/`: `tree_16S.aln` (só as 229) e o alinhamento com as sequências da turma, se os monitores o tiverem colocado lá. Todos aparecem no visualizador da próxima seção.


In [4]:
# Como alinhar com o MAFFT
style = {"description_width": "150px"}

# =========================
# INTERFACE
# =========================

input_file = widgets.Text(
    value=str(COMBINED),
    description="FASTA de entrada:",
    style=style,
    layout=widgets.Layout(width="800px")
)

output_folder = widgets.Text(
    value=str(RESULT_DIR),
    description="Pasta de saída:",
    style=style,
    layout=widgets.Layout(width="800px")
)

output_name = widgets.Text(
    value=MAFFT_OUT.name,
    description="Nome do arquivo:",
    style=style,
    layout=widgets.Layout(width="800px")
)

threads = widgets.BoundedIntText(
    value=4,
    min=1,
    max=64,
    description="Threads:",
    style=style
)

strategy = widgets.Dropdown(
    options=[
        ("Automática (recomendado)", "--auto"),
        ("FFT-NS-2 (rápido)", "--retree 2"),
        ("FFT-NS-1 (mais rápido)", "--retree 1"),
        ("L-INS-i (mais preciso, lento)", "--localpair"),
        ("G-INS-i (global, lento)", "--globalpair")
    ],
    value="--auto",
    description="Estratégia:",
    style=style,
    layout=widgets.Layout(width="450px")
)

button = widgets.Button(
    description="Quero meu alinhamento!",
    button_style="primary",
    icon="play",
    layout=widgets.Layout(width="230px", height="45px")
)

output = widgets.Output()

# =========================
# EXECUÇÃO
# =========================

def run_mafft(b):
    button.disabled = True

    with output:
        output.clear_output(wait=True)

        try:
            infile = Path(input_file.value.strip()).expanduser()
            outdir = Path(output_folder.value.strip()).expanduser()
            outfile = outdir / output_name.value.strip()

            if not infile.is_file():
                print(f"Arquivo de entrada não encontrado: {infile}")
                print("Rode a célula 'Juntando as sequências do grupo' ou use um alinhamento pré-computado.")
                return

            if not output_name.value.strip() or Path(output_name.value.strip()).name != output_name.value.strip():
                print("Informe um nome válido para o arquivo de saída.")
                return

            if infile.resolve() == outfile.resolve():
                print("O arquivo de saída não pode ser igual ao de entrada.")
                return

            outdir.mkdir(parents=True, exist_ok=True)

            cmd = ["mafft"]

            if strategy.value == "--localpair":
                cmd += ["--localpair", "--maxiterate", "1000"]
            elif strategy.value == "--globalpair":
                cmd += ["--globalpair", "--maxiterate", "1000"]
            elif strategy.value == "--retree 2":
                cmd += ["--retree", "2"]
            elif strategy.value == "--retree 1":
                cmd += ["--retree", "1"]
            else:
                cmd += ["--auto"]

            cmd += ["--thread", str(threads.value), str(infile)]

            print("Alinhando sequências com MAFFT...")
            print("Comando:", " ".join(cmd))

            with outfile.open("w") as f:
                subprocess.run(
                    cmd,
                    stdout=f,
                    stderr=subprocess.PIPE,
                    text=True,
                    check=True
                )

            print("\n✓ Alinhamento concluído!")
            print(f"Arquivo: {outfile.resolve()}")

        except subprocess.CalledProcessError as e:
            print(f"Erro na execução do MAFFT:\n{e.stderr}")
            if "outfile" in locals():
                outfile.unlink(missing_ok=True)

        except FileNotFoundError:
            print("MAFFT não encontrado. Verifique se está instalado e no PATH,")
            print("ou use o MAFFT online / um alinhamento pré-computado.")
            if "outfile" in locals():
                outfile.unlink(missing_ok=True)

        except Exception as e:
            print(f"Erro: {e}")

        finally:
            button.disabled = False


button.on_click(run_mafft)

display(
    widgets.VBox([
        widgets.HTML("<h3>Alinhamento de sequências — MAFFT</h3>"),
        input_file,
        output_folder,
        output_name,
        widgets.HBox([strategy, threads]),
        button,
        output
    ])
)

## Vendo os alinhamentos

Escolha um alinhamento e clique em **Mostrar alinhamento**:

- **Mapa de gaps** mostra o alinhamento inteiro: azul é nucleotídeo, laranja é gap.
- **Resíduos (janela)** mostra as bases de um trecho; escolha o início e a largura.
- **Destacar** coloca no topo, em vermelho, as sequências cujo nome contém o texto digitado (o que estiver escrito depois do ">" e antes do primeiro espaço).

A lista inclui os arquivos pré-computados em `dados/` e o que vocês gerarem em `resultados/`. Depois de rodar o MAFFT ou o trimAl, clique em **Atualizar lista**.


In [5]:
# Visão geral
display(visualizador(lambda: arquivos_de_alinhamento(DATA_DIR, RESULT_DIR)))

Notem as regiões quase vazias no alinhamento! O gene 16S tem cerca de 1.500 nt, mas o alinhamento é mais longo porque inclui inserções presentes em poucas sequências e regiões difíceis de comparar entre todas as espécies.

## Removendo regiões com trimAl

O **trimAl** remove colunas do alinhamento segundo o critério escolhido. Indique o alinhamento de entrada (o seu, em `resultados/`, ou um pré-computado em `dados/`). Depois, volte ao visualizador, clique em **Atualizar lista** e compare antes e depois.


In [6]:
# Rodando o TrimAL
style = {"description_width": "150px"}

# =========================
# INTERFACE
# =========================

input_trimal = widgets.Text(
    value=str(MAFFT_OUT),
    description="Alinhamento:",
    style=style,
    layout=widgets.Layout(width="800px")
)

output_folder_trimal = widgets.Text(
    value=str(RESULT_DIR),
    description="Pasta de saída:",
    style=style,
    layout=widgets.Layout(width="800px")
)

output_name_trimal = widgets.Text(
    value="16S_com_isolado.trim.aln",
    description="Nome do arquivo:",
    style=style,
    layout=widgets.Layout(width="800px")
)

method_trimal = widgets.Dropdown(
    options=[
        ("Automático (recomendado)", "automated1"),
        ("Gappyout", "gappyout"),
        ("Strict", "strict"),
        ("Strictplus", "strictplus"),
        ("No gaps", "nogaps"),
        ("Sem trimming", "none")
    ],
    value="automated1",
    description="Método:",
    style=style,
    layout=widgets.Layout(width="450px")
)

button_trimal = widgets.Button(
    description="Quero meu alinhamento trimado!",
    button_style="primary",
    icon="scissors",
    layout=widgets.Layout(width="280px", height="45px")
)

output_trimal = widgets.Output()

# =========================
# EXECUÇÃO
# =========================

def run_trimal(b):
    button_trimal.disabled = True

    with output_trimal:
        output_trimal.clear_output(wait=True)

        try:
            infile = Path(input_trimal.value.strip()).expanduser()
            outdir = Path(output_folder_trimal.value.strip()).expanduser()
            filename = output_name_trimal.value.strip()

            if not infile.is_file():
                print(f"Alinhamento não encontrado: {infile}")
                return

            if not filename or Path(filename).name != filename:
                print("Informe um nome válido para o arquivo de saída.")
                return

            outfile = outdir / filename

            if infile.resolve() == outfile.resolve():
                print("O arquivo de saída não pode ser igual ao de entrada.")
                return

            outdir.mkdir(parents=True, exist_ok=True)

            metodo = method_trimal.value

            print("Removendo regiões do alinhamento com trimAl...")

            if metodo == "none":
                shutil.copyfile(infile, outfile)
                print("Nenhum trimming aplicado. Arquivo copiado.")
            else:
                cmd = [
                    "trimal",
                    "-in", str(infile),
                    "-out", str(outfile),
                    f"-{metodo}"
                ]

                print("Comando:", " ".join(cmd))

                subprocess.run(
                    cmd,
                    check=True,
                    capture_output=True,
                    text=True
                )

            print("\n✓ Processamento concluído!")
            print(f"Arquivo: {outfile.resolve()}")

        except FileNotFoundError as e:
            print(f"Arquivo ou programa não encontrado: {e}")

        except subprocess.CalledProcessError as e:
            print(f"Erro na execução do trimAl:\n{e.stderr}")
            if "outfile" in locals():
                outfile.unlink(missing_ok=True)

        except Exception as e:
            print(f"Erro: {e}")

        finally:
            button_trimal.disabled = False


button_trimal.on_click(run_trimal)

display(
    widgets.VBox([
        widgets.HTML("<h3>Remoção de regiões do alinhamento — trimAl</h3>"),
        input_trimal,
        output_folder_trimal,
        output_name_trimal,
        method_trimal,
        button_trimal,
        output_trimal
    ])
)

## Gerando a árvore: IQ-TREE

Com o alinhamento pronto, a árvore é inferida por máxima verossimilhança no **IQ-TREE**, pelo servidor web ([iqtree.cibiv.univie.ac.at](http://iqtree.cibiv.univie.ac.at)):

1. Envie o alinhamento aparado (`resultados/16S_com_isolado.trim.aln`).
2. Deixe a seleção automática de modelo (**ModelFinder**) e peça **1000 réplicas de UFBoot**.
3. Quando terminar, baixe o arquivo `.treefile` e salve em `resultados/` com o nome **`16S_isolado.tree.treefile`**.

Na linha de comando, o equivalente é:

```bash
iqtree -s resultados/16S_com_isolado.trim.aln -m MFP -B 1000 -T AUTO --prefix 16S_isolado.tree
```

Com ~230 sequências isso leva em torno de 15–20 minutos. Se o servidor demorar demais ou falhar, os monitores fornecem a árvore pronta com o mesmo nome. Com o arquivo no lugar, a opção **16S + isolado da turma** aparece no notebook 03.

**Pergunta:** qual alinhamento deveríamos usar para o IQ-TREE, o original ou o aparado? O que se ganha e o que se perde ao remover colunas?
